# University Timetable Clash Detector Using Python
**Anindita Mukherjee** | BWU/BCA/25/159 | BCA, CSS | Brainware University | Guide: Jayanta Aich

Run the cells from top to bottom.

## 1. Program code

In [1]:
"""University Timetable Clash Detector - Anindita Mukherjee, BWU/BCA/25/159, BCA (CSS), Brainware University. Guide: Jayanta Aich"""
import csv
import sys
from itertools import combinations

REQUIRED = ["course", "teacher", "room", "group", "day", "start", "end"]


def to_minutes(t):
    h, m = map(int, t.strip().split(":"))
    return h * 60 + m


def load_classes(filename):
    classes = []
    with open(filename, newline="") as f:
        reader = csv.DictReader(f)
        missing = [c for c in REQUIRED if c not in (reader.fieldnames or [])]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
        for n, row in enumerate(reader, start=2):
            try:
                row["start"] = to_minutes(row["start"])
                row["end"] = to_minutes(row["end"])
            except ValueError:
                print(f"Line {n}: bad time format, skipped.")
                continue
            if row["end"] <= row["start"]:
                print(f"Line {n}: end time not after start, skipped.")
                continue
            classes.append(row)
    return classes


def overlaps(a, b):
    return a["day"] == b["day"] and a["start"] < b["end"] and b["start"] < a["end"]


def find_clashes(classes):
    clashes = []
    for a, b in combinations(classes, 2):
        if not overlaps(a, b):
            continue
        reasons = []
        if a["group"] == b["group"]:
            reasons.append("same student group")
        if a["teacher"] == b["teacher"]:
            reasons.append("same teacher")
        if a["room"] == b["room"]:
            reasons.append("same room")
        if reasons:
            clashes.append((a, b, reasons))
    return clashes


def save_clashes(clashes, filename="clashes.csv"):
    with open(filename, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["day", "class_1", "class_2", "reason"])
        for a, b, r in clashes:
            w.writerow([a["day"], a["course"], b["course"], "; ".join(r)])


def main(filename='data/timetable.csv'):
    clashes = find_clashes(load_classes(filename))
    if not clashes:
        print("No clashes found.")
        return
    print(f"{len(clashes)} clash(es) found:\n")
    for a, b, r in clashes:
        print(f"{a['day']}: {a['course']} vs {b['course']} -> {', '.join(r)}")
    save_clashes(clashes)
    print("\nSaved to clashes.csv")




## 2. Sample timetable data

In [2]:
sample = """course,teacher,room,group,day,start,end
Maths,Dr. Sen,R101,CSE-A,Mon,09:00,10:00
Physics,Dr. Roy,R102,CSE-A,Mon,09:30,10:30
Python,Dr. Das,R101,CSE-B,Mon,09:45,10:45
Chemistry,Dr. Sen,R103,CSE-C,Mon,10:00,11:00
English,Dr. Pal,R104,CSE-A,Tue,11:00,12:00
Maths Lab,Dr. Sen,R105,CSE-B,Tue,11:30,12:30
"""
with open("timetable.csv", "w") as f:
    f.write(sample)
print("timetable.csv created")

timetable.csv created


## 3. Find clashes

In [3]:
main('timetable.csv')

2 clash(es) found:

Mon: Maths vs Physics -> same student group
Mon: Maths vs Python -> same room

Saved to clashes.csv


## 4. Tests

In [4]:
# Tests
from itertools import combinations
def c(course, day, s, e, teacher="T", room="R", group="G"):
    return dict(course=course, day=day, start=to_minutes(s), end=to_minutes(e), teacher=teacher, room=room, group=group)
assert overlaps(c("A","Mon","09:00","10:00"), c("B","Mon","09:30","10:30"))
assert not overlaps(c("A","Mon","09:00","10:00"), c("B","Mon","10:00","11:00"))
assert not overlaps(c("A","Mon","09:00","10:00"), c("B","Tue","09:00","10:00"))
assert overlaps(c("A","Mon","09:00","10:00"), c("B","Mon","09:00","10:00"))
a=c("A","Mon","09:00","10:00",teacher="X",room="R1",group="G1"); b=c("B","Mon","09:30","10:30",teacher="X",room="R2",group="G2")
assert find_clashes([a,b])[0][2]==["same teacher"]
print("All 5 tests passed")

All 5 tests passed
